# bartz benchmark — faststochtree DGP

Times bartz on the same simulated DGP used in the faststochtree C++ benchmarks.
Intended to run on a Colab GPU.

**DGP:** $y = \sin(2\pi x_1) + \varepsilon$, $\varepsilon \sim \mathcal{N}(0, \sigma^2)$, $X \sim U[0,1]^{n \times p}$

Each of the `n_iters` iterations uses a different random seed, so timing and RMSE
are averaged over independent data realisations.

**JAX note:** the first iteration includes XLA/JIT compilation overhead.
Per-iteration times are reported so you can see the warmup effect; the summary
prints both the full average and the average excluding iteration 1.

In [ ]:
%pip install bartz

Configuration — edit `n_iters` to control how many independent runs are averaged:

In [ ]:
n_train    = 50_000   # training points
n_test     = 500      # test points
p          = 50       # number of predictors
num_trees  = 200      # number of trees (ntree)
n_burnin   = 200      # burn-in iterations (nskip)
n_samples  = 1_000    # posterior samples (ndpost)
sigma_true = 1.0      # true error std
seed       = 12345    # base RNG seed — iteration i uses key(seed + i)
n_iters    = 10       # number of independent runs to average over

Run bartz `n_iters` times, each with an independent data and model seed:

In [ ]:
from time import perf_counter
from bartz.BART import gbart
from jax import numpy as jnp, random

times_ms = []
rmses    = []
sigmas   = []

print(f'bartz benchmark  n_train={n_train:,}  n_test={n_test}  p={p}  trees={num_trees}')
print(f'burnin={n_burnin}  samples={n_samples}  iters={n_iters}  sigma_true={sigma_true}\n')

for i in range(n_iters):
    # Independent data and model seeds for each iteration.
    # data seeds:  seed+0, seed+1, ..., seed+n_iters-1
    # model seeds: seed+n_iters, ..., seed+2*n_iters-1  (no overlap)
    key_data, key_noise = random.split(random.key(seed + i), 2)
    key_model = random.key(seed + n_iters + i)

    # Simulate data — bartz uses (p, n) layout
    n_total = n_train + n_test
    X_all   = random.uniform(key_data, shape=(p, n_total))
    y_all   = jnp.sin(2 * jnp.pi * X_all[0]) + sigma_true * random.normal(key_noise, shape=(n_total,))
    X_train = X_all[:, :n_train]
    y_train = y_all[:n_train]
    X_test  = X_all[:, n_train:]
    y_test  = y_all[n_train:]

    print(f'iter {i+1:2d}/{n_iters}: sampling...', end='', flush=True)

    start = perf_counter()
    bart_model = gbart(
        X_train,
        y_train,
        ntree      = num_trees,
        nskip      = n_burnin,
        ndpost     = n_samples,
        printevery = n_burnin + n_samples + 1,  # suppress per-iteration output
        seed       = key_model,
    )
    elapsed_ms = int(round((perf_counter() - start) * 1000))

    yhat_test = bart_model.predict(X_test)           # (n_samples, n_test)
    yhat_mean = jnp.mean(yhat_test, axis=0)
    rmse  = float(jnp.sqrt(jnp.mean(jnp.square(yhat_mean - y_test))))
    sigma = float(jnp.sqrt(jnp.mean(jnp.square(bart_model.sigma))))

    jit_note = '  ← includes JIT compilation' if i == 0 else ''
    print(f'  {elapsed_ms:8,} ms  RMSE={rmse:.4f}  sigma={sigma:.4f}{jit_note}')

    times_ms.append(elapsed_ms)
    rmses.append(rmse)
    sigmas.append(sigma)

Summary:

In [ ]:
import numpy as np

avg_ms    = int(np.mean(times_ms))
avg_ms_ex = int(np.mean(times_ms[1:]))  # excluding first (JIT warmup)
avg_rmse  = np.mean(rmses)
avg_sigma = np.mean(sigmas)

print(f'Average over {n_iters} iterations:')
print(f'  time:  {avg_ms:,} ms  (excl. iter 1: {avg_ms_ex:,} ms)')
print(f'  RMSE:  {avg_rmse:.4f}')
print(f'  sigma: {avg_sigma:.4f}')